## Overview

 This notebook is intended to:
 * Show how to connect to Awesense's Energy Data Model (EDM) using REST API.
 * Use the REST API endpoints to retrieve the following:
     * Information about active grids
     * Information about grid elements
     * Tracing information from specific grid element
     * Time series for different grid elements 
* The data returned by the REST API is in JSON format. To extract and display the information more clearly, different formats such as 'json_format', and 'csv_download' are used in the notebook. The 'json_format' is the default and works all the time. It displays a data frame using the first-level JSON objects, which can be further expanded. 

The notebook uses the Awesense api-connect gateway: https://api-account.awesense.com/. The gateway contains additional documentation regarding the endpoints used in this notebook, their available parameters and output options. Please refer to it for more information. 

It is assumed that the user has been given access to it and has the necessary credentials. Otherwise, please contact us at [api@awesense.com](api@awesense.com).

---

## Set up

In [1]:
import getpass
import requests
import pandas as pd
import base64

pd.set_option('display.max_columns', None)

### Helper Functions

In [2]:
def get_rest_api(url: str, key: str, auth_key: str, format: str, params: dict = None, element_id: str = None, csv_file_prefix: str = None) -> pd.DataFrame:
    """
    The function calls a REST API endpoint with the appropriate parameters. It handles any errors that may occur during the request
    and returns the results in a data frame format. The returned data frame can be further unpacked to extract specific information as needed. 

    Parameters:
    -url: The URL of the REST API endpoint.
    -key: The subscription key (primary or secondary) from the `Profile` page on the api-connect gateway website. 
    -auth_key: The basic authentication credentials.
    -format: The format in which the request URLs provide the response. REST endpoints return differently formatted JSONs and/or binary (CSV) files.
    -params: A dictionary that may contain additional parameters to pass to the API endpoint. By default, it is set to None.
    -element_id: Name of an element to include in the name of a CSV file. By default, it is set to None.
    -csv_file_prefix: Prefix to include as part of the CSV file name. By default, it is set to None.
    
    Returns:
    -df: The results as a pandas data frame. 
    """
    # Create the request headers
    hdr = {
        'Cache-Control': 'no-cache',
        'Ocp-Apim-Subscription-Key': key,
        'Authorization': auth_key,
    }
    
    try:
        # Make the GET request to the API endpoint
        if params:
            request = requests.get(url, headers=hdr, params=params)
        else:
            request = requests.get(url, headers=hdr)
            
        # Check the response status code
        if request.status_code == 200:
            # Process the response based on the specified format
            if format == 'json_format':
                # convert the returned json object to a data frame
                df = pd.json_normalize(request.json())
            elif format == 'csv_download':
                with open(element_id + '_' + csv_file_prefix + '_csv_output.csv', 'wb') as file:
                    file.write(request.content)
                    print("CSV file downloaded successfully.")
                # Read the CSV file into a Pandas data frame
                print('Below is a display of the CSV file.')
                df = pd.read_csv(element_id + '_' + csv_file_prefix + '_csv_output.csv')
            else:
                print('This is not the correct format. Possible formats are: json_format, and csv_download.')
                df = None
        else:
            print(f'Failed to download. Server response: {request.status_code}')
            print('Response content:', request.text)  # Print the response content for more details
            df = None
        
    except requests.exceptions.RequestException as exceptions:
        print(f'An error occurred while making the HTTP request: {exceptions}')
        df = None
    except Exception as exceptions:
        print(f'An unexpected error occurred: {exceptions}')
        df = None
    
    return df


def unpack_json(df: pd.DataFrame, column_to_unpack: str, columns_to_append: list = None) -> pd.DataFrame:
    """
    Unpacks the list of dictionaries returned by the `get_rest_api` function when used with the `json_format` option. 
    Optionally display additional `columns_to_append` in a data frame. Returns the results in a data frame. 

    Parameters:
    - df: A data frame containing the list of dictionaries.
    - column_to_unpack: The name of the column containing the list of dictionaries.
    - columns_to_append: A list of additional columns to display. The default is None. 

    Return:
    - df_unpacked: The results as a pandas data frame.
    """

    # Create an empty list that will store the results.
    unpacked_dicts = []
    
    for _, row in df.iterrows():
            
            # Unpack the dictionary in the current row.
            unpack = pd.json_normalize(row[column_to_unpack])
            
            if columns_to_append is not None:
                # Loop over the list of columns to be displayed.
                for column in columns_to_append:
                    unpack[column] = row[column]
            
            # Append the unpacked dictionary to the list.
            unpacked_dicts.append(unpack)
        
    # Concatenate all the unpacked dictionaries into a single data frame.
    df_unpacked = pd.concat(unpacked_dicts, ignore_index=True)

    return df_unpacked

### Connection Variables

In [3]:
# Enter the user_name, and password that you use to log into the server.
server_user_name = getpass.getpass(prompt='Username: ')

In [4]:
server_password = getpass.getpass(prompt='Password: ')

In [5]:
# Enter the subscription key (primary or secondary) from the `Profile` page on the api-connect gateway website. 
# If you don't have a subscription key, you will need to create one. 
# To do so, go to the `Products` page on the api-connect gateway website, click on the desired product name, enter a product description, and click `Subscribe`. 
subscription_key = getpass.getpass(prompt='Subscription Key: ')

In [6]:
# Create BasicAuth credential based on user_name and password.  
auth_str = server_user_name + ':' + server_password
byte_str = auth_str.encode('ascii')
encoded_data = base64.b64encode(byte_str)
basic_auth = 'Basic ' + str(encoded_data, encoding='utf-8')

# Delete the credential variables for security purposes.
del server_user_name, server_password, auth_str, byte_str

---

## Basic Data Retrieval

### Grids

In [7]:
# Get all the grids. 
url_grid = 'https://api-connect.awesense.com/basic_data_retrieval/api/v1/grid'
grids = get_rest_api(url_grid, subscription_key, basic_auth, format = 'json_format')
grids

,active,description,id
0,True,awefice,awefice
1,True,North Central Zone,North Central Zone


### Grid Elements

In [8]:
# Get all the `grid_elements` of a specific type. 
# Types include ACLineSegment, Battery, Busbar, Cabinet, Capacitor, CircuitBreaker, Disconnector, 
# Enclosure, EVCharger, FixedSensor, Fuse, grid_data_source, Jumper, Manhole, Meter, Photovoltaic,
# Pole, ServiceDeliveryPoint, SpokeLine, Substation, Switch, Transformer, UnmeteredLoad.   
# The following example gets information about type 'Meter'.
grid_id = 'awefice'
grid_element_type = 'Meter'
url_grid_explorer = 'https://api-connect.awesense.com/basic_data_retrieval/api/v1/grid_explorer'

params = {
        'limit': '30',
        'offset': '0', 
        'ordering': '{"direction": "ASC", "query_name": "grid_id"}',
        'filter': f'{{"operator": "=", "column_name": "grid_id", "value": "{grid_id}"}}',
        'export': 'false',
        'grid_element_type': grid_element_type,
}

grid_elements = get_rest_api(url_grid_explorer, subscription_key, basic_auth, format = 'json_format', params=params)
grid_elements

,headers,results,counts.filtered,counts.total
0,"[{'filterable': True, 'list': False, 'name': '...","[{'address': 'Theresa Brown, 449 Catherine Pra...",27,20994


In [9]:
# The above data frame contains lists of dictionaries that must be unpacked for viewing the results.
# Unpack the `results` column to view all the meters.
results = unpack_json(grid_elements, 'results')
results.head()

,address,business_identifier_id,commission_date,connected_to_lvmv,description,energy_source,estimated,grid_id,hvmv_parent_element,id,interval,is_consumer,is_critical,is_operation_status,is_producer,label,latitude,longitude,lv_feeder,make_model,maximal_demand,maximal_demand_units,meter_multiplier,meter_number,meter_type,mv_feeder,name,parent_transformer_id,phases,read_type,replacement_cycle,sdp_id,service_drop_type,source_rating_kw,standard_profile_id,tariff_id,type,type_of_consumer,underground,voltage_level,zone_level_1,zone_level_2,zone_level_3,zone_level_4
0,"Theresa Brown, 449 Catherine Prairie, South Da...",None,None,None,awefice,None,None,awefice,None,m_1,3600.0,True,False,True,False,m_1,49.267484,-123.105894,None,None,30.0,A,None,V3-5516213X,amm,None,None,transformer_16,A,interval,None,None,None,None,None,res_basic,Meter,residential,None,LV,None,None,None,awegrid
1,"Larry Barnes, 040 Evan Court, Reidhaven, PW 93312",None,None,None,awefice,None,None,awefice,None,m_10,3600.0,True,False,True,False,m_10,49.266986,-123.103169,None,None,50.0,A,None,E1-5252081B,amm,None,None,transformer_36,ABC,interval,None,None,None,None,None,comm_green,Meter,business,None,LV,None,None,None,awegrid
2,"Wendy Hall, 93995 Cohen Harbors Apt. 206, Mart...",None,None,None,awefice,None,None,awefice,None,m_11,3600.0,True,False,True,False,m_11,49.267313,-123.101544,None,None,30.0,A,None,Y8-5397150A,amm,None,None,transformer_43,ABC,interval,None,None,None,None,None,comm_green,Meter,industrial,None,LV,None,None,None,awegrid
3,"Brett White, 87748 Lewis Trace, Lake Linda, NE...",None,None,None,awefice,None,None,awefice,None,m_12,3600.0,True,False,True,False,m_12,49.266717,-123.102841,None,None,150.0,A,None,M7-2558866S,amm,None,None,transformer_47,ABC,interval,None,None,None,None,None,comm_ev,Meter,business,None,LV,None,None,None,awegrid
4,"Kevin Keith, USNS Knapp, FPO AP 63389",None,None,None,awefice,None,None,awefice,None,m_13,3600.0,True,False,True,False,m_13,49.266862,-123.105948,None,None,30.0,A,None,J2-3548166G,amm,None,None,transformer_31,B,interval,None,None,None,None,None,res_basic,Meter,residential,None,LV,None,None,None,awegrid


In [10]:
# Display all the Meter grid element IDs.
results.id

0      m_1
1     m_10
2     m_11
3     m_12
4     m_13
5     m_14
6     m_15
7     m_16
8     m_17
9     m_18
10    m_19
11     m_2
12    m_20
13    m_21
14    m_22
15    m_23
16    m_24
17    m_25
18    m_26
19    m_27
20     m_3
21     m_4
22     m_5
23     m_6
24     m_7
25     m_8
26     m_9
Name: id, dtype: object

In [11]:
# Similar API as in the previous cell, but with a different grid element type. Notice that the resulting columns are different. 
# Please consult the grid schema CSV file in the online documentation available from the TGI help menu to learn more 
# information about all the different properties of the various grid element types.
grid_id = 'awefice'
grid_element_type = 'Photovoltaic'
url_grid_explorer = 'https://api-connect.awesense.com/basic_data_retrieval/api/v1/grid_explorer'

params = {
        'limit': '10',
        'offset': '0', 
        'ordering': '{"direction": "ASC", "query_name": "grid_id"}',
        'filter': f'{{"operator": "=", "column_name": "grid_id", "value": "{grid_id}"}}',
        'export': 'false',
        'grid_element_type': grid_element_type,
}

grid_elements_pv = get_rest_api(url_grid_explorer, subscription_key, basic_auth, format = 'json_format', params=params)
grid_elements_pv

,headers,results,counts.filtered,counts.total
0,"[{'filterable': True, 'list': False, 'name': '...","[{'ac_size': 3651.0, 'azimuth_angle': 170.0, '...",6,79


In [12]:
# Unpack the `results` column to view the elements.
results = unpack_json(grid_elements_pv, 'results')
results.head()

,ac_size,azimuth_angle,commission_date,dc_size,dc_to_ac_ratio,degradation_25_years,degradation_annual,description,generation_capacity,grid_id,id,inverter_make_model,is_consumer,is_producer,label,latitude,longitude,lv_feeder,meter_id,name,number_of_panels,ownership,panel_area,panel_make_model,panel_rated_power,parent_transformer_id,phases,racking,sdp_id,tilt_angle,type,underground,voltage_level
0,3651.0,170.0,2014-08-16,4440.0,1.216,7.50,0.30,awefice,3651.0,awefice,pv_10,SolarEdge HDWave,True,True,pv_10,49.266986,-123.103169,None,m_10,pv_10,12.0,private,1.727200,LG Neon R [LG370Q1C-V5],370.0,transformer_36,ABC,roof,None,42.0,Photovoltaic,None,LV
1,24954.0,171.0,2019-02-04,29970.0,1.201,6.25,0.25,awefice,24954.0,awefice,pv_11,SMA\tSunny Boy,True,True,pv_11,49.267313,-123.101544,None,m_11,pv_11,81.0,private,1.748536,Panasonic EverVolt [EVPV370],370.0,transformer_43,ABC,roof,None,41.0,Photovoltaic,None,LV
2,1813.0,174.0,2014-07-26,2075.0,1.144,6.25,0.25,awefice,1813.0,awefice,pv_19,Fronius Primo,True,True,pv_19,49.267908,-123.105328,None,m_19,pv_19,5.0,private,1.915056,SunPower Maxeon3 [SPR-MAX6-415-BLK-E3-AC],415.0,transformer_63,ABC,carport,None,39.0,Photovoltaic,None,LV
3,49984.0,172.0,2018-03-20,62530.0,1.251,6.25,0.25,awefice,49984.0,awefice,pv_24,SMA\tSunny Boy,True,True,pv_24,49.268284,-123.101677,None,m_24,pv_24,169.0,private,1.748536,Panasonic EverVolt [EVPV370],370.0,transformer_92,ABC,ground,None,40.0,Photovoltaic,None,LV
4,3651.0,180.0,2012-08-26,4440.0,1.216,7.50,0.30,awefice,3651.0,awefice,pv_27,SolarEdge HDWave,True,True,pv_27,49.267876,-123.103364,None,m_27,pv_27,12.0,private,1.727200,LG Neon R [LG370Q1C-V5],370.0,transformer_92,A,carport,None,38.0,Photovoltaic,None,LV


In [13]:
# Display all the Photovoltaic grid element IDs.
results.id

0    pv_10
1    pv_11
2    pv_19
3    pv_24
4    pv_27
5     pv_4
Name: id, dtype: object

### Details about a grid element

In [14]:
# Get details about specific grid elements. 
grid_id = 'awefice'
grid_element = 'm_6'
url_grid_element = f'https://api-connect.awesense.com/basic_data_retrieval/api/v1/grid/{grid_id}/element/{grid_element}'

params = {
        'all_sources': 'true'
}

grid_element_details = get_rest_api(url_grid_element, subscription_key, basic_auth, format = 'json_format', params=params )
grid_element_details

,ambiguous,customer_type,grid_data_sources,grid_id,id,is_consumer,is_producer,is_switchable,phases,score,snapped_point,terminal1_cn,terminal2_cn,type,underground,meta.address,meta.business_identifier_id,meta.commission_date,meta.connected_to_lvmv,meta.energy_source,meta.estimated,meta.hvmv_parent_element,meta.interval,meta.is_critical,meta.is_operation_status,meta.label,meta.latitude,meta.longitude,meta.lv_feeder,meta.make_model,meta.maximal_demand,meta.maximal_demand_units,meta.meter_multiplier,meta.meter_number,meta.meter_type,meta.mv_feeder,meta.name,meta.parent_transformer_id,meta.read_type,meta.replacement_cycle,meta.sdp_id,meta.service_drop_type,meta.source_rating_kw,meta.standard_profile_id,meta.tariff_id,meta.type_of_consumer,meta.voltage_level,meta.zone_level_1,meta.zone_level_2,meta.zone_level_3,meta.zone_level_4
0,False,meter,"[{'friendly_id': 'm_6', 'id': '3b61660d-9b7a-4...",awefice,m_6,True,False,False,C,None,"[-123.103424204188, 49.2674386639061]",con_70,con_70_t2,Meter,None,"Michael Macias, 031 Autumn Causeway Suite 931,...",None,None,None,None,None,None,3600,False,True,m_6,None,None,None,None,30,A,None,L5-4986193Z,amm,None,None,transformer_26,interval,None,None,None,None,None,res_basic,residential,LV,None,None,None,awegrid


In [15]:
# Unpack the `grid_data_source` column containing the list of dictionaries.
# The grid_data_source ID represents the linking between time-series data and physical elements on the grid. 
# The ID is used further down in this notebook to retrieve the time series. 
unpack_json(grid_element_details, 'grid_data_sources', ['customer_type', 'meta.address'])

,friendly_id,id,metrics,phases,provider,type,valid,customer_type,meta.address
0,m_6,3b61660d-9b7a-442b-ae54-2ca1039f98f3,[V],C,awe_voltage_reads,CONSUMER,"[2021-01-01T08:01:00.000000Z,)",meter,"Michael Macias, 031 Autumn Causeway Suite 931,..."
1,m_6,84e32b3f-1cf6-4b8a-9a55-f59ec574f8c1,[kWh],C,awe_meter_reading,CONSUMER,"[2021-01-01T08:00:00.000000Z,)",meter,"Michael Macias, 031 Autumn Causeway Suite 931,..."


### Grid Tracing

In [16]:
# The example below demonstrates the tracing of transformers from a meter in an upstream direction. The results are saved to a CSV file. 
grid_id = 'awefice'
grid_element_id = 'm_6'
end_point_name = 'upstream_tracing'
url_grid_trace = 'https://api-connect.awesense.com/basic_data_retrieval/api/v1/grid/trace'

params = {
        'trace_type': 'upstream',
        'output_type': 'transformer_download',
        'grid_id': grid_id,
        'grid_element_id': grid_element_id,
}

grid_element_trace = get_rest_api(url_grid_trace, subscription_key, basic_auth, format = 'csv_download', params=params, element_id=grid_element_id, csv_file_prefix=end_point_name)
grid_element_trace

CSV file downloaded successfully.
Below is a display of the CSV file.


,ID,Phases,commission_date,enclosure_id,hvmv_parent_element,installation_type,is_operation_status,label,latitude,load_loss,longitude,maintenance_type,make_model,manufacture_date,mv_feeder,name,no_load_loss,ownership,parent_transformer_id,primary_voltage,rating_kva,secondary_voltage,voltage_level,winding_configuration
0,transformer_2,ABC,1989-01-01,enc_1,NaN,station,True,tr_2,NaN,246932,NaN,oil,NaN,NaN,NaN,tr_2,24775,utility,NaN,200000,20000.0,12500,HV/MV,NaN
1,transformer_26,C,1969-01-01,NaN,NaN,pole,True,tr_26,NaN,312,NaN,oil,NaN,NaN,NaN,tr_26,58,utility,NaN,12500,10.0,240,MV/LV,NaN


In [17]:
# The example below demonstrates the tracing from transformers in a downstream direction.  
# For more information about different types of traces and output sources, please refer to the documentation in the API-connect gateway.
grid_id = 'awefice'
grid_element_id = 'transformer_2'
url_grid_trace = 'https://api-connect.awesense.com/basic_data_retrieval/api/v1/grid/trace'

params = {
        'trace_type': 'downstream',
        'output_type': 'summary',
        'grid_id': grid_id,
        'grid_element_id': grid_element_id,
}

grid_element_trace = get_rest_api(url_grid_trace, subscription_key, basic_auth, format = 'json_format', params=params)
grid_element_trace

,consumers,consumers_breakdown,consumers_phases,meters,meters_breakdown,meters_phases,producer_elements,producer_phases,transformers_phases,producer_breakdown.Battery.ABC,producer_breakdown.Battery.C,producer_breakdown.Battery.sum,producer_breakdown.EVCharger.A,producer_breakdown.EVCharger.ABC,producer_breakdown.EVCharger.sum,producer_breakdown.Photovoltaic.ABC,producer_breakdown.Photovoltaic.C,producer_breakdown.Photovoltaic.sum,producer_breakdown.Transformer.ABC,producer_breakdown.Transformer.sum,producer_phase_total.A,producer_phase_total.ABC,producer_phase_total.C,producer_phase_total.sum,transformers_breakdown.count.A,transformers_breakdown.count.ABC,transformers_breakdown.count.B,transformers_breakdown.count.C,transformers_breakdown.count.sum,transformers_breakdown.kVA.A,transformers_breakdown.kVA.ABC,transformers_breakdown.kVA.B,transformers_breakdown.kVA.C,transformers_breakdown.kVA.sum
0,15,"[{'A': 3, 'ABC': 3, 'B': 3, 'C': 3, 'sum': 12,...","[A, B, C, ABC]",15,"[{'A': 3, 'ABC': 6, 'B': 3, 'C': 3, 'sum': 15,...","[A, B, C, ABC]",9,"[A, C, ABC]","[A, B, C, ABC]",1,1,2,1,2,3,2,1,3,1,1,1,6,2,9,1,4,1,1,7,5.0,20275.0,30.0,10.0,20320.0


In [18]:
# Unpack the `consumers_breakdown` column containing the list of dictionaries. 
unpack_json(grid_element_trace, 'consumers_breakdown')

,A,ABC,B,C,sum,type
0,3.0,3,3.0,3.0,12,Residential
1,NaN,2,NaN,NaN,2,Commercial
2,NaN,1,NaN,NaN,1,Industrial
3,3.0,6,3.0,3.0,15,Total


### Time Series

There are different endpoints for retrieving time series, each suited for different use cases. For more information, refer to the api-connect gateway documentation.

In [19]:
# Get time series data associated with a given grid data source.
grid_id = 'awefice'
grid_data_source = '84e32b3f-1cf6-4b8a-9a55-f59ec574f8c1' # From the details grid element above
url_grid_data_source = f'https://api-connect.awesense.com/basic_data_retrieval/api/v1/grid/grid_data_source/{grid_data_source}/data'

params = {
        'units': 'kWh',
        'group_by': 'hour',
        'start': '2021-01-01T08:00:00.000000Z',
        'end': '2021-03-01T08:00:00.000000Z',
        'max_points': '2000',
        'timezone': 'America/Vancouver',
        'include_sign': 'false',
        'export': 'false'
}

ts = get_rest_api(url_grid_data_source, subscription_key, basic_auth, format = 'json_format', params=params)
ts

,series,units
0,"[{'amount': 2.17, 'period': 3600.0, 'timestamp...",kWh


In [20]:
# Unpack the time series and include the `units` column. 
grid_element_ts = unpack_json(ts, 'series', ['units'])
grid_element_ts

,amount,period,timestamp,units
0,2.17,3600.0,2021-01-01T08:00:00.000000Z,kWh
1,2.06,3600.0,2021-01-01T09:00:00.000000Z,kWh
2,2.09,3600.0,2021-01-01T10:00:00.000000Z,kWh
3,1.99,3600.0,2021-01-01T11:00:00.000000Z,kWh
4,2.19,3600.0,2021-01-01T12:00:00.000000Z,kWh
...,...,...,...,...
1411,2.20,3600.0,2021-03-01T03:00:00.000000Z,kWh
1412,2.40,3600.0,2021-03-01T04:00:00.000000Z,kWh
1413,2.38,3600.0,2021-03-01T05:00:00.000000Z,kWh
1414,1.97,3600.0,2021-03-01T06:00:00.000000Z,kWh


In [21]:
# Get consumption data for any grid element with associated grid data sources of type CONSUMER. 
# This example exports the result to a CSV file. 
grid_id = 'awefice'
grid_element_id = 'm_6'
end_point_name = 'time_series'
url_grid_ts_consumer = f'https://api-connect.awesense.com/basic_data_retrieval/api/v1/grid/{grid_id}/element/{grid_element_id}/data'

params = {
        'units': 'kWh',
        'group_by': 'hour',
        'start': '2021-01-01T08:00:00.000000Z',
        'end': '2021-03-01T08:00:00.000000Z',
        'max_points': '2000',
        'timezone': 'America/Vancouver',
        'export': 'true'
}

grid_element_ts = get_rest_api(url_grid_ts_consumer, subscription_key, basic_auth,  format = 'csv_download', params=params, element_id=grid_element_id, csv_file_prefix=end_point_name)
grid_element_ts

CSV file downloaded successfully.
Below is a display of the CSV file.


,Id,Timestamp (UTC),kWh
0,m_6,2021-01-01 08:00:00+00:00,2.166
1,m_6,2021-01-01 09:00:00+00:00,2.062
2,m_6,2021-01-01 10:00:00+00:00,2.088
3,m_6,2021-01-01 11:00:00+00:00,1.993
4,m_6,2021-01-01 12:00:00+00:00,2.190
...,...,...,...
1411,m_6,2021-03-01 03:00:00+00:00,2.199
1412,m_6,2021-03-01 04:00:00+00:00,2.397
1413,m_6,2021-03-01 05:00:00+00:00,2.383
1414,m_6,2021-03-01 06:00:00+00:00,1.967


In [22]:
# Get any kind of time-series data for a grid element.
grid_id = 'North Central Zone'
grid_element_id = '7676'
url_grid_ts_consumer = f'https://api-connect.awesense.com/basic_data_retrieval/api/v1/grid/{grid_id}/element/{grid_element_id}/flexible_time_series_data'

params = {
        'units': 'kW',
        'group_by': 'dynamic',
        'start': '2021-01-01T08:00:00.000000Z',
        'end': '2021-04-01T08:00:00.000000Z',
        'phases': 'ABC',
        'max_points': '1000',
        'timezone': 'America/Vancouver',
        'use_last_value': 'false',
        'combine_phases': 'false',
        'data_source_type': 'SENSOR',
        'only_matching_units': 'false',
}

ts = get_rest_api(url_grid_ts_consumer, subscription_key, basic_auth,  format = 'json_format', params=params)
ts

,phase,series,units,source.friendly_id,source.grid_data_source_id
0,A,"[{'amount': 508.0, 'period': 10800.0, 'timesta...",kW,7676.A,1914cd98-0bac-47b2-84d7-25d4ab7ab532
1,B,"[{'amount': 233.0, 'period': 10800.0, 'timesta...",kW,7676.B,8a047ba3-33aa-460f-a86d-77ef572e0380
2,C,"[{'amount': 395.0, 'period': 10800.0, 'timesta...",kW,7676.C,fdf04446-1aac-4680-9c32-5ff8d681e4a8


In [23]:
# Unpack the time series for the different phases and include the `units`, and `source.friendly_id` columns.
phase_ts = unpack_json(ts, 'series', ['phase', 'units', 'source.friendly_id'])
phase_ts

,amount,period,timestamp,phase,units,source.friendly_id
0,508.0,10800.0,2021-01-01T08:00:00.000000Z,A,kW,7676.A
1,520.0,10800.0,2021-01-01T11:00:00.000000Z,A,kW,7676.A
2,526.0,10800.0,2021-01-01T14:00:00.000000Z,A,kW,7676.A
3,524.0,10800.0,2021-01-01T17:00:00.000000Z,A,kW,7676.A
4,532.0,10800.0,2021-01-01T20:00:00.000000Z,A,kW,7676.A
...,...,...,...,...,...,...
2158,395.0,10800.0,2021-03-31T19:00:00.000000Z,C,kW,7676.C
2159,410.0,10800.0,2021-03-31T22:00:00.000000Z,C,kW,7676.C
2160,390.0,10800.0,2021-04-01T01:00:00.000000Z,C,kW,7676.C
2161,472.0,10800.0,2021-04-01T04:00:00.000000Z,C,kW,7676.C


In [24]:
# Get per-phase time series data for each sensor (raptor or SCADA) associated with the given grid element.
grid_id = 'North Central Zone'
grid_element_id = '10680'
url_grid_ts_consumer = f'https://api-connect.awesense.com/basic_data_retrieval/api/v1/grid/{grid_id}/element/{grid_element_id}/chart_data'

params = {
        'units': 'kWh',
        'group_by': 'day',
        'start': '2021-01-01T08:00:00.000000Z',
        'end': '2021-04-01T08:00:00.000000Z',
        'max_points': '2000',
        'timezone': 'America/Vancouver',
        'include_sign': 'false'
}

ts = get_rest_api(url_grid_ts_consumer, subscription_key, basic_auth, format = 'json_format', params=params)
ts

,phase,series,units,sensor.grid_data_source_id,sensor.sensor_id
0,A,"[{'amount': 4550, 'period': 86400.0, 'timestam...",kWh,445366b4-6e05-448f-88fa-a920957aaa36,10680.A
1,B,"[{'amount': 7925, 'period': 86400.0, 'timestam...",kWh,581f6e63-883f-4580-9bf1-5032ca741dd9,10680.B
2,C,"[{'amount': 6309, 'period': 86400.0, 'timestam...",kWh,d24f5e30-c866-4e05-b4db-520c6793deb5,10680.C


In [25]:
# Unpack the time series for the different phases and add the `units` column. 
per_phase = unpack_json(ts, 'series', ['phase', 'units'])
per_phase

,amount,period,timestamp,phase,units
0,4550,86400.0,2021-01-01T08:00:00.000000Z,A,kWh
1,4522,86400.0,2021-01-02T08:00:00.000000Z,A,kWh
2,4720,86400.0,2021-01-03T08:00:00.000000Z,A,kWh
3,5017,86400.0,2021-01-04T08:00:00.000000Z,A,kWh
4,4990,86400.0,2021-01-05T08:00:00.000000Z,A,kWh
...,...,...,...,...,...
268,6376,86400.0,2021-03-28T07:00:00.000000Z,C,kWh
269,6108,86400.0,2021-03-29T07:00:00.000000Z,C,kWh
270,6383,86400.0,2021-03-30T07:00:00.000000Z,C,kWh
271,6215,86400.0,2021-03-31T07:00:00.000000Z,C,kWh


---

In [26]:
# Delete the credential variables for security purposes.
del subscription_key, basic_auth